# 第 17 章 · 因果 mask：不许偷看后面的词

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/17-causal-mask.html) · 仓库里 `transformer-deep-dive/notes/lessons/17-causal-mask.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 能看的格子 | 下三角（含对角线） | `torch.tril(torch.ones(3, 3))` | 左下三角是 1 |
| mask 表 M | 能看 0、要遮 −∞ | `torch.triu(torch.full((3, 3), -inf), diagonal=1)` | 右上 3 格是 −inf |
| 越负越接近 0 | $e^{-1}, e^{-10}, e^{-100}$ | `torch.exp` | 0.368、4.54e-05、3.78e-44 |
| $e^{-\infty}$ | 正好是 0 | `torch.exp(-inf)` | 0.0 |
| 加 M 再 softmax | $\text{softmax}(\cdot + M)$ | `torch.softmax(scores + M, -1)` | "咬"：[0.73, 0.27, 0] |
| 自带的因果 attention | 同上 | `F.scaled_dot_product_attention(..., is_causal=True)` | 和手写一样 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
import torch.nn.functional as F

**函数 `causal_mask`**：M：能看填 0，要遮填 −∞。torch.triu(..., diagonal=1) 取对角线右上方。

In [ ]:
def causal_mask(n: int) -> torch.Tensor:
    """M：能看填 0，要遮填 −∞。torch.triu(..., diagonal=1) 取对角线右上方。"""
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)

**函数 `masked_softmax`**：softmax(分数 + M)：mask 加在 softmax 之前。

In [ ]:
def masked_softmax(scores: torch.Tensor) -> torch.Tensor:
    """softmax(分数 + M)：mask 加在 softmax 之前。"""
    return torch.softmax(scores + causal_mask(scores.shape[-1]), dim=-1)

**函数 `r`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def r(t: torch.Tensor, n: int = 2) -> list[float]:
    return [round(float(x), n) for x in t]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("能看的格子（下三角）torch.tril:\n", torch.tril(torch.ones(3, 3, dtype=torch.bool)))
print("M =\n", causal_mask(3))
bite = torch.tensor([2.0, 1.0, 3.0])  # "咬"那一行：看狗、看咬、看人

In [ ]:
print("\n'咬'那一行 [2, 1, 3]")
print("  不遮       :", r(torch.softmax(bite, -1)), "← 67% 偷看了答案'人'")
for v in [-1.0, -10.0, -100.0]:
    print(f"  第 3 格改成 {v:>6}: e^x = {float(torch.exp(torch.tensor(v))):.3g}")
print("  e^(−∞)     =", float(torch.exp(torch.tensor(float("-inf")))))
print("  加 M 后    :", r(masked_softmax(torch.tensor([[0.0, 0, 0], bite.tolist(), [0, 0, 0]]))[1]))

In [ ]:
print("\n完整的 masked attention 和 PyTorch 自带的 is_causal=True 一样吗:")
torch.manual_seed(0)
q, k, v = (torch.randn(3, 4) for _ in range(3))
mine = masked_softmax(q @ k.T / 4**0.5) @ v
builtin = F.scaled_dot_product_attention(q[None], k[None], v[None], is_causal=True)[0]
print("  ", torch.allclose(mine, builtin, atol=1e-6))

In [ ]:
print("\n小心 1：PyTorch 的布尔 attn_mask 里 True = 能看（和'遮住'正好反过来）")
allowed = torch.tril(torch.ones(3, 3, dtype=torch.bool))
same = F.scaled_dot_product_attention(q[None], k[None], v[None], attn_mask=allowed)[0]
print("   attn_mask=下三角 True 结果一样:", torch.allclose(same, builtin, atol=1e-6))
print(
    "小心 2：一整行全是 −∞，softmax 得 nan（没有能看的词）:",
    torch.softmax(torch.full((3,), float("-inf")), -1).tolist(),
)

## 3. 代码里才看得到的

1. **PyTorch 布尔 mask 里 True = 能看**：`F.scaled_dot_product_attention` 的 `attn_mask` 传布尔表时，True 表示"允许看"，和"遮住"的说法正好反过来。传反了不报错，结果却全错，最容易踩坑。
2. **一整行全是 −∞，softmax 得 nan**：分母是 0 ÷ 0。因果 mask 永远保留对角线（自己看自己），所以不会出现；但以后自己拼 mask（比如再遮掉补齐用的空位）时要小心。
3. **−inf 写作 `float("-inf")`**：和论文 L237 的 −∞ 一样，是一个真正的"负无穷"，不是很大的负数。

## 4. 自己改一改

- 把上面任意一格里的数字改一改，再按 Shift + Enter，看结果怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 一次说一个，后面是答案，要遮住；
> 分数表遮右上角，能看填 0，要遮填 −∞；
> e 的 −∞ 次方是 0，剩下的重新分成一。